# Lexicon maintenance - member-form review (phase 8)

Thin driver; the logic is in `lang_tools.lexicon.ingestion.form_review` and
`lang_tools.llm.runner`.

Non-English member forms of five words or more, or over 40 characters, are mostly
legitimate (proper names, multiword terms), but some are a definition pasted in as
a word, text in the wrong language, or a garbled string. That is a judgement per
form, so this is propose-for-review:

1. **Worklist** - `suspicious_forms`, one item per (lemma, concept) sense.
2. **Propose** - the `form_review` chain classifies each form; the verdicts become
   `proposed` `FormDrop` rows in `data/curated/member_form_drops.jsonl`.
3. **Review** - flip each row to `accepted` (drop it) or `rejected` (keep it).
   Watch for real names the model reads as definitions, and for two forms joined
   by "et" / "ou" (those want splitting, not dropping).
4. **Rebuild** - `apply_form_drops` runs first in the build, so nothing downstream
   sees a dropped form.

The worklist is small (under 1,000 forms), so the live path is fine; keep the
sample-first rule anyway.

**Paid-run rule.** Every LLM job here follows the same order: run a small live sample,
read its measured usage and the extrapolated cost, compare models if the choice is
open, and only then flip `RUN_PAID` and submit the full batch. Nothing in this notebook
spends money until that flag is set by hand.

In [ ]:
import collections

from llm_core.chat.config.openai import ChatOpenAIConfig

from lang_tools.lexicon.ingestion.form_review import MEMBER_FORM_DROPS_FILE
from lang_tools.lexicon.ingestion.form_review import FormDrop
from lang_tools.lexicon.ingestion.form_review import suspicious_forms
from lang_tools.lexicon.maintenance import curated_path
from lang_tools.lexicon.maintenance import read_jsonl
from lang_tools.lexicon.maintenance import write_jsonl
from lang_tools.llm.form_review import build_form_review_chain
from lang_tools.llm.runner import load_checkpoint
from lang_tools.llm.runner import prompt_sha
from lang_tools.llm.runner import read_records
from lang_tools.llm.runner import run_chain
from lang_tools.llm.runner import summarize_usage
from lang_tools.params.lang_tools_params import get_lang_tools_params
from lang_tools.params.load_env import load_env

load_env()
data_fol = get_lang_tools_params().paths.data_fol
work = data_fol / "_raw" / "lexicon" / "maintenance" / "form_review"
curated = curated_path(data_fol, MEMBER_FORM_DROPS_FILE)

MODEL = "gpt-6-luna"
#: Live (standard tier) prices per 1M tokens for the chosen model.
PRICES = {"input_per_m": 0.10, "cached_per_m": 0.01, "output_per_m": 0.50}
RUN_PAID = False  # flip by hand only after reading the sample's measured cost


class NoReasoningConfig(ChatOpenAIConfig):
    """ChatOpenAIConfig with reasoning off (llm-core has no kwargs passthrough)."""

    reasoning_effort: str = "none"


chain = build_form_review_chain(NoReasoningConfig(model=MODEL))

## Worklist

In [ ]:
items = dict(suspicious_forms(data_fol))
len(items), collections.Counter(item.language for item in items.values())

## Sample, then the full run (live)

In [ ]:
checkpoint = work / "run.jsonl"
run_chain(chain, items.items(), checkpoint=checkpoint, limit=20)
totals = summarize_usage(read_records(checkpoint))
totals, f"estimated full run, live: ${totals.cost(**PRICES) * len(items) / max(totals.records, 1):.2f}"

In [ ]:
if RUN_PAID:
    run_chain(chain, items.items(), checkpoint=checkpoint)

## Propose

Only the forms the chain wants dropped become rows. Rows already reviewed in the
curated file are kept as they are.

In [ ]:
sha = prompt_sha(chain)
reviewed = {row.key: row for row in read_jsonl(FormDrop, curated)} if curated.exists() else {}
for key, out in load_checkpoint(checkpoint, chain.output_model).items():
    if not out.drop or key in reviewed:
        continue
    lemma_id, ckey = key.split(":")
    reviewed[key] = FormDrop(
        lemma_id=lemma_id,
        concept_key=ckey,
        language=items[key].language,
        form=items[key].form,
        kind=out.kind,
        drop=out.drop,
        reason=out.reason,
        model=MODEL,
        prompt_sha=sha,
    )
write_jsonl(sorted(reviewed.values(), key=lambda row: row.key), curated)
collections.Counter(row.status for row in reviewed.values())

## Review

Edit `data/curated/member_form_drops.jsonl`: `accepted` drops the form,
`rejected` keeps it. Then rebuild (`02_transform`) and re-run the quality gate.